# Representação dos dados do agente em Vetores e Matrizes

**Projeto:** Agente virtual da FECAP para atendimento acadêmico

**Objetivo do agente:** cruzar os documentos oficiais da FECAP com os dados individuais de cada aluno (notas, faltas, disciplinas) para responder automaticamente perguntas como *"qual é a minha média?"* ou *"estou em risco de reprovação?"*, sem que o aluno precise ir até a secretaria.

Neste notebook mostramos:

1. Quais colunas da base `Historico.xlsx` viram **vetores** e **matrizes**;
2. Duas operações de Álgebra Linear (produto escalar/combinação linear e norma/distância) aplicadas a esses dados — mais uma operação extra (ângulo/similaridade de cosseno);
3. A implementação em Python;
4. O que cada resultado significa para o funcionamento do agente.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

df = pd.read_excel('Historico.xlsx', sheet_name='Sheet1')

print("Dimensões do dataset:", df.shape)
print("Colunas:", list(df.columns))


Dimensões do dataset: (274901, 13)
Colunas: ['ID_ALUNO', 'Ano/Semestre Letivo', 'Período Acadêmico de Oferta', '% Assiduidade', '% de Faltas', '% Nota Obtida', 'Aulas Ministradas', 'Nota Distribuída', 'Disciplina', 'Situação', 'Nota Obtida', 'Faltas Lançadas', 'ARQUIVO_ORIGEM']


## 1. Quais dados viram vetores e matrizes

A base tem **274.901 registros** (um registro = um aluno cursando uma disciplina em um semestre) e **9.815 alunos**.

Para o agente conseguir responder perguntas sobre *um* aluno, cada aluno-semestre é representado como um **vetor**, cujas posições são as disciplinas cursadas naquele semestre:

- **v_nota** → vetor com `% Nota Obtida` em cada disciplina;
- **v_carga** → vetor com `Aulas Ministradas` (carga horária) de cada disciplina, usado como *peso*;
- **v_falta** → vetor com `% de Faltas` em cada disciplina.

Quando olhamos para **vários alunos ao mesmo tempo**, esses vetores empilhados formam uma **matriz Aluno × Disciplina** — exatamente a estrutura que o agente consulta em memória para responder qualquer aluno instantaneamente, em vez de alguém folhear históricos em papel na secretaria.

Como exemplo concreto, vamos usar o aluno `ID_ALUNO = 2` no semestre `2020-2`, que teve disciplinas aprovadas e uma reprovada (bom exemplo para testar a detecção de risco).


In [2]:
aluno_id = 2
semestre = '2020-2'

sub = df[(df.ID_ALUNO == aluno_id) & (df['Ano/Semestre Letivo'] == semestre)].reset_index(drop=True)
print(sub[['Disciplina','% Nota Obtida','% de Faltas','Aulas Ministradas','Situação']])

disciplinas = sub['Disciplina'].tolist()
v_nota  = sub['% Nota Obtida'].to_numpy(dtype=float)
v_carga = sub['Aulas Ministradas'].to_numpy(dtype=float)
v_falta = sub['% de Faltas'].to_numpy(dtype=float)

print()
print("v_nota  =", v_nota)
print("v_carga =", v_carga)
print("v_falta =", v_falta)


                               Disciplina  % Nota Obtida  % de Faltas  Aulas Ministradas   Situação
0            Fundamentos de Macroeconomia           60.0        10.53                 76   Aprovado
1       Direito Empresarial e do Trabalho           83.0         0.00                 74   Aprovado
2  Estatística Descritiva e Probabilidade           79.5         0.00                 74   Aprovado
3              Administração de Marketing           81.5         0.00                 76   Aprovado
4   Introdução à Administração Financeira           51.5         0.00                 36  Reprovado
5        Técnicas de Pesquisa em Negócios           85.8         0.00                 36   Aprovado

v_nota  = [60.  83.  79.5 81.5 51.5 85.8]
v_carga = [76. 74. 74. 76. 36. 36.]
v_falta = [10.53  0.    0.    0.    0.    0.  ]


## 2. Operação 1 — Produto escalar / Combinação linear: média ponderada

A secretaria calcula a média do semestre ponderando cada nota pela carga horária da disciplina (disciplinas com mais aulas pesam mais no resultado final). Isso é exatamente uma **combinação linear** de `v_nota`, usando `v_carga` como pesos, normalizada pela soma dos pesos:

$$\text{média ponderada} = \frac{v\_nota \cdot v\_carga}{\sum v\_carga}$$

O numerador é o **produto escalar** entre os dois vetores.


In [3]:
media_simples = v_nota.mean()
media_ponderada = np.dot(v_nota, v_carga) / v_carga.sum()

print(f"Média simples das notas:                 {media_simples:.2f}%")
print(f"Média ponderada pela carga horária:       {media_ponderada:.2f}%")


Média simples das notas:                 73.55%
Média ponderada pela carga horária:       74.52%


**O que isso significa para o agente:** com uma única operação vetorial, o agente devolve ao aluno a média oficial do semestre (74,52%), já ponderada corretamente pela carga horária de cada disciplina — o mesmo cálculo que hoje é feito manualmente na secretaria. Note que a média simples (73,55%) seria uma resposta *diferente e incorreta* do ponto de vista institucional, o que mostra por que o agente precisa usar o vetor de pesos (`v_carga`) e não apenas uma média aritmética ingênua.


## 3. Operação 2 — Vetores de margem, Norma e Distância Euclidiana: detecção de risco

A FECAP (como a maioria das IES no Brasil) exige, para aprovação: **nota mínima de 60%** e **frequência mínima de 75%** (ou seja, no máximo **25% de faltas**). Chamamos esse par de valores de **vetor-regra institucional**.

Para cada disciplina, construímos o **vetor de margem** do aluno em relação à regra:

- `margem_nota  = nota% − 60`  → positivo = folga acima do mínimo; negativo = abaixo do mínimo (risco);
- `margem_falta = 25 − falta%` → positivo = dentro do limite; negativo = excedeu o limite (risco).

A **norma** (comprimento) do vetor `[margem_nota, margem_falta]` mede a distância euclidiana entre a situação real do aluno e o ponto-limite `(60, 25)` definido pela regra — quanto menor a norma e mais negativos os componentes, mais perto (ou dentro) da zona de risco o aluno está.


In [4]:
NOTA_MINIMA = 60.0
FALTA_MAXIMA = 25.0

margem_nota  = v_nota - NOTA_MINIMA
margem_falta = FALTA_MAXIMA - v_falta

print("margem_nota  =", np.round(margem_nota, 2))
print("margem_falta =", np.round(margem_falta, 2))
print()

for d, mn, mf in zip(disciplinas, margem_nota, margem_falta):
    dist = np.linalg.norm([mn, mf])
    status = "RISCO (nota abaixo do mínimo)" if mn < 0 else "OK"
    print(f"{d:42s} margem_nota={mn:6.2f}  margem_falta={mf:6.2f}  ||margem||={dist:6.2f}  -> {status}")

print()
dist_geral = np.linalg.norm(v_nota - NOTA_MINIMA)
print(f"Distância euclidiana de v_nota até o vetor-regra (60 em cada disciplina): {dist_geral:.2f}")


margem_nota  = [ 0.  23.  19.5 21.5 -8.5 25.8]
margem_falta = [14.47 25.   25.   25.   25.   25.  ]

Fundamentos de Macroeconomia               margem_nota=  0.00  margem_falta= 14.47  ||margem||= 14.47  -> OK
Direito Empresarial e do Trabalho          margem_nota= 23.00  margem_falta= 25.00  ||margem||= 33.97  -> OK
Estatística Descritiva e Probabilidade     margem_nota= 19.50  margem_falta= 25.00  ||margem||= 31.71  -> OK
Administração de Marketing                 margem_nota= 21.50  margem_falta= 25.00  ||margem||= 32.97  -> OK
Introdução à Administração Financeira      margem_nota= -8.50  margem_falta= 25.00  ||margem||= 26.41  -> RISCO (nota abaixo do mínimo)
Técnicas de Pesquisa em Negócios           margem_nota= 25.80  margem_falta= 25.00  ||margem||= 35.93  -> OK

Distância euclidiana de v_nota até o vetor-regra (60 em cada disciplina): 45.93


**O que isso significa para o agente:** a disciplina *"Introdução à Administração Financeira"* tem `margem_nota = -8.5`, ou seja, o vetor do aluno está **abaixo** do vetor-regra nessa dimensão — o agente identifica isso automaticamente e pode responder *"Você foi reprovado em Introdução à Administração Financeira por nota, faltando 8,5 pontos percentuais para o mínimo"*. Nas demais disciplinas as margens são positivas, então o agente responde que o aluno está regular nelas. A distância euclidiana geral (45,93) resume, em um único número, o quão distante o semestre do aluno está do "pior caso possível" (nota mínima em tudo) — útil para o agente priorizar quais alunos monitorar de perto.


## 4. Operação extra — Ângulo / Similaridade de cosseno com a turma

Além de comparar o aluno com a regra fixa da instituição, o agente pode comparar o **padrão** de desempenho do aluno com a média da turma nas mesmas disciplinas, usando o ângulo entre os vetores `[nota, falta]` de cada um (via similaridade de cosseno). Um ângulo pequeno indica um padrão parecido com o da turma; um ângulo grande indica um comportamento atípico (ex.: notas altas mas faltas muito acima da média, ou vice-versa).


In [5]:
turma = df[(df['Ano/Semestre Letivo'] == semestre) & (df['Disciplina'].isin(disciplinas))]
media_turma = turma.groupby('Disciplina')[['% Nota Obtida','% de Faltas']].mean().reindex(disciplinas)

v_aluno_turma = sub[['% Nota Obtida','% de Faltas']].to_numpy(dtype=float).flatten()
v_media_turma = media_turma.to_numpy(dtype=float).flatten()

cos_sim = np.dot(v_aluno_turma, v_media_turma) / (np.linalg.norm(v_aluno_turma) * np.linalg.norm(v_media_turma))
angulo = np.degrees(np.arccos(np.clip(cos_sim, -1, 1)))

print(f"Similaridade de cosseno entre o aluno e a média da turma: {cos_sim:.4f}")
print(f"Ângulo entre os vetores: {angulo:.2f} graus")


Similaridade de cosseno entre o aluno e a média da turma: 0.9937
Ângulo entre os vetores: 6.46 graus


**O que isso significa para o agente:** o ângulo de ~6,5° (similaridade de 0,99) indica que, apesar da reprovação pontual, o padrão geral do aluno é bem próximo do padrão médio da turma — ou seja, não é um caso atípico que exija uma resposta totalmente diferente do agente. Se o ângulo fosse grande (similaridade baixa), o agente poderia sinalizar esse aluno como um caso fora do padrão, sugerindo encaminhamento a um atendimento humano em vez de uma resposta automática.


## 5. Matriz Aluno × Disciplina

Por fim, juntando os vetores de vários alunos que cursaram as mesmas disciplinas no mesmo semestre, obtemos a **matriz Aluno × Disciplina** (linhas = alunos, colunas = disciplinas, valores = `% Nota Obtida`). É essa matriz — e não arquivos separados por aluno — que permite ao agente localizar instantaneamente a linha de qualquer aluno e responder sua pergunta, em vez de alguém procurar manualmente pasta por pasta na secretaria.


In [6]:
base_matriz = df[(df['Ano/Semestre Letivo'] == semestre) & (df['Disciplina'].isin(disciplinas))]
amostra_alunos = base_matriz['ID_ALUNO'].unique()[:8]
if aluno_id not in amostra_alunos:
    amostra_alunos = np.append(amostra_alunos, aluno_id)

matriz_df = base_matriz[base_matriz['ID_ALUNO'].isin(amostra_alunos)]
M = matriz_df.pivot_table(index='ID_ALUNO', columns='Disciplina', values='% Nota Obtida')

print("Matriz Aluno x Disciplina (amostra, % Nota Obtida):")
print(M.round(1))
print()
print("Formato da matriz:", M.shape)


Matriz Aluno x Disciplina (amostra, % Nota Obtida):
Disciplina  Administração de Marketing  Direito Empresarial e do Trabalho  Estatística Descritiva e Probabilidade  Fundamentos de Macroeconomia  Introdução à Administração Financeira  \
ID_ALUNO                                                                                                                                                                                 
2                                 81.5                               83.0                                    79.5                          60.0                                   51.5   
2232                               NaN                               83.8                                    83.5                          84.7                                    NaN   
3115                              69.3                                NaN                                     NaN                           NaN                                   70.0   
4684              

**O que isso significa para o agente:** cada linha dessa matriz é o "cartão de identidade acadêmico" de um aluno; valores `NaN` simplesmente indicam que aquele aluno não cursou aquela disciplina naquele semestre (matriz esparsa, comum em dados reais). É sobre essa estrutura que todas as operações acima (produto escalar, norma, distância, cosseno) são aplicadas linha a linha — permitindo que o agente escale o mesmo cálculo para os quase 10 mil alunos da base, e não apenas para o exemplo mostrado aqui.

## Conclusão

Com poucas operações de Álgebra Linear — **produto escalar/combinação linear**, **norma/distância euclidiana** e **ângulo/similaridade de cosseno** — aplicadas sobre vetores de notas, faltas e carga horária, o agente consegue automatizar exatamente o tipo de cálculo e verificação que hoje depende de um atendimento presencial na secretaria: calcular médias oficiais, verificar se o aluno está acima ou abaixo das regras institucionais, e identificar padrões atípicos que merecem atenção humana.
